In [10]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"
ADAPTER_DIR = "/mnt/data1/Ryunho/DenseRewardRLHF-PPO/exp_ppo_3/models"

INPUT_CSV = "/mnt/data1/Ryunho/DenseRewardRLHF-PPO/final_8.csv"
OUTPUT_CSV = "/mnt/data1/Ryunho/DenseRewardRLHF-PPO/final_8_weird.csv"

NUM_SAMPLES = 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

df = pd.read_csv(INPUT_CSV, encoding='cp949')

COLUMN_NAME = "x (Input prompt)"

if COLUMN_NAME not in df.columns:
    raise ValueError(f"CSV 파일에 '{COLUMN_NAME}' 컬럼이 없습니다. 컬럼 이름을 확인하세요.")

df = df.dropna(subset=[COLUMN_NAME])

sampled = df.sample(n=NUM_SAMPLES, random_state=42).reset_index(drop=True)
prompts = sampled[COLUMN_NAME].tolist()

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    load_in_4bit=True
)

print("Loading PPO LoRA adapter...")
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR
)
model.eval()

model.config.use_cache = False
base_model.config.use_cache = False

def generate_from_prompt(
    prompt: str,
    max_new_tokens: int = 900,
    temperature: float = 0.8,
    top_p: float = 0.9
) -> str:
    
    messages = [{"role": "user", "content": prompt}]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(chat_text, return_tensors="pt")
    input_ids = inputs["input_ids"].to(DEVICE)
    attention_mask = inputs["attention_mask"].to(DEVICE)

    input_len = input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False
        )

    gen_ids = outputs[0, input_len:]
    decoded = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    for prefix in ["Assistant:", "assistant:", "assistant", "Assistant"]:
        if decoded.startswith(prefix):
            decoded = decoded[len(prefix):].lstrip()

    return decoded

results = []

print(f"총 {NUM_SAMPLES}개의 프롬프트에 대해 생성 시작...")
for i, prompt in enumerate(prompts):
    print(f"[{i+1}/{NUM_SAMPLES}]")
    print("Input prompt preview:", str(prompt)[:80].replace("\n", " "), "...")

    output = generate_from_prompt(prompt)

    if "[INPUT ARTICLE]" in output or "[ROLE]" in output:
        print("⚠ 경고: generated_article 안에 프롬프트 메타 태그가 포함되어 있습니다.")

    results.append({
        "x (Input prompt)": prompt, 
        "generated_article": output
    })


out_df = pd.DataFrame(results)
out_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print("\n🎉 완료! 결과가 저장됨:", OUTPUT_CSV)


Loading tokenizer...


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 19f0ba5b-f84c-4502-92e8-2fc53691420e)')' thrown while requesting HEAD https://huggingface.co/microsoft/Phi-3.5-mini-instruct/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


Loading base model...


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.29s/it]


Loading PPO LoRA adapter...
총 1개의 프롬프트에 대해 생성 시작...
[1/1]
Input prompt preview: [ROLE] You are a professional journalist writing for a trusted news organization ...

🎉 완료! 결과가 저장됨: /mnt/data1/Ryunho/DenseRewardRLHF-PPO/final_8_weird.csv


In [ ]:
# filename 예: compute_reward_stats_news.py

import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from peft import PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"
RM_ADAPTER_DIR = "/mnt/data1/Ryunho/DenseRewardRLHF-PPO/exp/final_model"  # 네 reward model LoRA 디렉토리
PROMPT_DATASET = "RyunHo/news_prompt"
N_SAMPLES = 256          # 통계용으로 몇 개만 봄
MAX_NEW_TOKENS = 256     # 너무 길게 말고

print("Load tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

print("Load base LM for RM...")
from transformers import AutoModelForCausalLM
base_lm = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    load_in_4bit=True,
)

print("Load RM adapter...")
rm = PeftModel.from_pretrained(base_lm, RM_ADAPTER_DIR)
rm.eval().to(DEVICE)

if hasattr(rm, "config"):
    rm.config.use_cache = False

print("Load prompts...")
ds = load_dataset(PROMPT_DATASET, split="train")
prompts = [ds[i]["prompt"] for i in range(min(N_SAMPLES, len(ds)))]

all_rewards = []

@torch.no_grad()
def get_reward_for_text(text: str):
    # RM이 학습에 쓴 포맷(peak segmentation) 그대로 맞춰야 가장 좋지만
    # 여기서는 “하나의 답변 전체에 대한 score”만 대략 보려고 simplification
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    out = rm(**inputs, use_cache=False)
    # denserlhf RM 구현에 따라 이름이 다를 수 있음: score / value / logits 등
    # 보통 shape: [batch, seq_len] 혹은 [batch]
    score = out["score"] if isinstance(out, dict) and "score" in out else out.logits.mean(-1)
    return score.mean().item()

print("Sampling rewards...")
for p in prompts:
    # 간단히: "prompt + RM이 대충 만들어낸 답변" 대신,
    # 지금은 RM이 prompt 자체에 대해 어떤 score를 주는지만 봐도
    # 스케일 감 잡는 데는 충분
    r = get_reward_for_text(p)
    all_rewards.append(r)

import numpy as np
all_rewards = np.array(all_rewards)
print("num samples:", len(all_rewards))
print("reward mean:", all_rewards.mean())
print("reward std:", all_rewards.std())


Load tokenizer...
Load base LM for RM...


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.31s/it]


Load RM adapter...
Load prompts...


Using the latest cached version of the dataset since RyunHo/news_prompt couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/fbsgh9847/.cache/huggingface/datasets/RyunHo___news_prompt/default/0.0.0/2e3fdabb0ccef00dc1c2ace58b3b639ebe8c0b29 (last modified on Tue Nov 25 20:52:36 2025).


Sampling rewards...
num samples: 256
reward mean: 32.7060686647892
reward std: 0.6641171207567439


: 